In [1]:
!pip install timm torch torchvision

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 19.3 MB/s  0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   --- ------------------------------------ 11.5/123.0 MB 56.7 MB/s eta 0:00:02
   ------- -------------------------------- 23.9/123.0 MB 57.9 MB/s eta 0:00:02
   ----------- ---------------------------- 36.2/123.0 MB 59.0 MB/s eta 0:00:02
   ---------------- ----------------------- 49.3/123.0 MB 59.0 MB/s eta 0:00:02
   -------------------- ------------------- 62.1/123.0 MB 59.6 MB/s eta 0:00:02
   ------------------------ --------------- 75.8/123.0 MB 60.9 MB/s eta 0:00:01
   ---------------------------- ----------- 88.3/123.0 MB 61.0 MB/s eta 0:00:01
   -------------------------------- ------- 99.6/123.0 MB 59.8 MB/s eta 0:00:01
   ---------------------------------- ---- 110.1/123.0 MB 58.9 MB/s eta 0:00:01
   ------------------------------------ -- 115.6/123.0 MB 55.6 MB/

In [2]:
import timm

# This downloads the pretrained CaiT weights to your local ~/.cache/huggingface folder
model = timm.create_model('cait_xxs24_224', pretrained=True)
print("CaiT model successfully downloaded and cached locally!")

C:\Users\qtm\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CaiT model successfully downloaded and cached locally!


C:\Users\qtm\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\qtm\.cache\huggingface\hub\models--timm--cait_xxs24_224.fb_dist_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
import os
import torch
import timm
from PIL import Image
import torchvision.transforms as transforms

# Force offline mode for the underlying HuggingFace hubs
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("Environment set to offline mode.")

Environment set to offline mode.


In [4]:
# Create the model using the cached weights
model_name = 'cait_xxs24_224' # Using the ultra-lightweight 12M parameter CaiT variant
model = timm.create_model(model_name, pretrained=True)
model.eval()  # Set the model to evaluation (inference) mode

# Move the model to your GPU if available to make it faster
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Successfully loaded {model_name} offline onto {device}!")

Successfully loaded cait_xxs24_224 offline onto cpu!


In [5]:
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

# Auto-resolve the exact transformation configurations (Resize to 224x224, ImageNet normalization)
config = resolve_data_config({}, model=model)
transform = create_transform(**config)

print("Automatically configured Image Transform pipeline:")
print(transform)

Automatically configured Image Transform pipeline:
Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)


In [6]:
# 1. Load your local image file
image_path = "bird.jpg"  
img = Image.open(image_path).convert('RGB')

# 2. Apply the preprocessing transforms and add a batch dimension
input_tensor = transform(img).unsqueeze(0).to(device) # Shape becomes [1, 3, 224, 224]

# 3. Pass the image forward through the CaiT network
with torch.no_grad():
    output = model(input_tensor)

# 4. Convert the raw logits into probability scores
probabilities = torch.nn.functional.softmax(output[0], dim=0)

print("Inference completed successfully!")

Inference completed successfully!


In [16]:
# 1. Extract the raw visual features from CaiT (bypassing the 1,000-class classifier)
with torch.no_grad():
    # This outputs the rich, internal token embeddings from the Class-Attention layer
    raw_visual_features = model.forward_features(input_tensor)

print("True Image Feature Shape from CaiT:", raw_visual_features.shape)

True Image Feature Shape from CaiT: torch.Size([1, 197, 192])


In [7]:
# Extract the top 5 indices and probabilities
top5_prob, top5_class_ids = torch.topk(probabilities, 5)

print("\n--- TOP 5 OFFLINE PREDICTIONS ---")
for i in range(top5_prob.size(0)):
    class_id = top5_class_ids[i].item()
    confidence = top5_prob[i].item() * 100
    print(f"Rank {i+1} | Class ID: {class_id:<4} | Confidence: {confidence:.2f}%")


--- TOP 5 OFFLINE PREDICTIONS ---
Rank 1 | Class ID: 92   | Confidence: 23.87%
Rank 2 | Class ID: 10   | Confidence: 23.61%
Rank 3 | Class ID: 14   | Confidence: 14.28%
Rank 4 | Class ID: 11   | Confidence: 7.01%
Rank 5 | Class ID: 95   | Confidence: 5.60%


In [15]:
# Sort all 1,000 probabilities from highest to lowest
all_probabilities, all_class_ids = torch.sort(probabilities, descending=True)

print(f"--- ALL {len(all_class_ids)} CLASS PREDICTIONS ---")
print(f"{'Rank':<6} | {'Class ID':<8} | {'Confidence':<12}")
print("-" * 32)

# Loop through all 1,000 predictions without any text labels
for i in range(all_class_ids.size(0)):
    class_id = all_class_ids[i].item()
    confidence = all_probabilities[i].item() * 100
    
    print(f"{i+1:<6} | {class_id:<8} | {confidence:.5f}%")

--- ALL 1000 CLASS PREDICTIONS ---
Rank   | Class ID | Confidence  
--------------------------------
1      | 92       | 23.86889%
2      | 10       | 23.61210%
3      | 14       | 14.27800%
4      | 11       | 7.01238%
5      | 95       | 5.60420%
6      | 12       | 4.64196%
7      | 16       | 3.85883%
8      | 15       | 2.25116%
9      | 19       | 1.12402%
10     | 94       | 0.60455%
11     | 91       | 0.56854%
12     | 13       | 0.52000%
13     | 20       | 0.41290%
14     | 17       | 0.17967%
15     | 964      | 0.15631%
16     | 86       | 0.13895%
17     | 90       | 0.09858%
18     | 277      | 0.09402%
19     | 928      | 0.08082%
20     | 483      | 0.07961%
21     | 417      | 0.07834%
22     | 733      | 0.07649%
23     | 78       | 0.07503%
24     | 979      | 0.07345%
25     | 927      | 0.06607%
26     | 885      | 0.06531%
27     | 741      | 0.06190%
28     | 21       | 0.06174%
29     | 454      | 0.06171%
30     | 967      | 0.06036%
31     | 962      | 0.0545

In [13]:
import torch
import torch.nn as nn

# 1. Store your 1,000 probabilities from the loop into a single tensor variable
# Ensure it has a batch dimension [1, 1000] for the MLP
mlp_input = probabilities.unsqueeze(0).to(device)


# 2. Define the 4-layer MLP Architecture
class FeatureReducerMLP(nn.Module):
    def __init__(self):
        super(FeatureReducerMLP, self).__init__()
        self.mlp = nn.Sequential(
            # Layer 1: 1000 -> 512
            nn.Linear(1000, 512),
            nn.ReLU(),
            
            # Layer 2: 512 -> 256
            nn.Linear(512, 256),
            nn.ReLU(),
            
            # Layer 3: 256 -> 128
            nn.Linear(256, 128),
            nn.ReLU(),
            
            # Layer 4: 128 -> 64 (Your target features)
            nn.Linear(128, 64)
        )
        
    def forward(self, x):
        return self.mlp(x)


# 3. Initialize the MLP model completely offline
mlp_reducer = FeatureReducerMLP().to(device)
mlp_reducer.eval() # Set to evaluation mode since we aren't training it yet


# 4. Pass the stored 1000-dimensional variable through the network
with torch.no_grad():
    reduced_features = mlp_reducer(mlp_input)


# 5. Extract the final 64 features into a clean flat list/variable
# .squeeze(0) removes the batch dimension, .tolist() converts it to standard Python numbers
final_64_features = reduced_features.squeeze(0).tolist()


# 6. Print the structural results and the first few features to verify
print("=== FEATURE REDUCTION COMPLETE ===")
print(f"Input Variable Shape  : {mlp_input.shape} (Batch Size, Classes)")
print(f"Output Variable Shape : {reduced_features.shape} (Batch Size, Reduced Features)")
print(f"Total features extracted into variable: {len(final_64_features)}")
print(final_64_features)

=== FEATURE REDUCTION COMPLETE ===
Input Variable Shape  : torch.Size([1, 1000]) (Batch Size, Classes)
Output Variable Shape : torch.Size([1, 64]) (Batch Size, Reduced Features)
Total features extracted into variable: 64
[0.050704117864370346, 0.052274253219366074, 0.02165350876748562, -0.005644824355840683, 0.06594865769147873, 0.05797934904694557, -0.00860423594713211, 0.05158174782991409, -0.011702459305524826, -0.08372226357460022, -0.06472655385732651, -0.002328605391085148, -0.042372897267341614, 0.047069840133190155, -0.04923174902796745, 0.0030665909871459007, -0.03615178167819977, -0.0424485057592392, 0.05759209394454956, 0.022327914834022522, -0.04246748238801956, -0.013188408687710762, 0.0018891915678977966, -0.029149994254112244, 0.03993649035692215, -0.012666694819927216, 0.030273478478193283, -0.0572749525308609, -0.018147161230444908, -0.05821679159998894, -0.05725172534584999, -0.05311032012104988, 0.031332723796367645, 0.08131508529186249, 0.0718168094754219, 0.0335077

In [17]:
# 1. Take your final_64_features and turn them back into a PyTorch tensor
# We add the batch dimension back so it's ready for PyTorch: shape [1, 64]
features_64_tensor = torch.tensor(final_64_features).unsqueeze(0).to(device)

# 2. Define a clean 64 -> 40 linear mapping layer
# This acts as the final connector to your data re-uploading circuit
quantum_mapper = nn.Linear(64, 40).to(device)
quantum_mapper.eval() # Set to evaluation mode

# 3. Pass the 64 features through to get exactly 40
with torch.no_grad():
    features_40_tensor = quantum_mapper(features_64_tensor)

# 4. Strip the batch dimension to make it a flat list for PennyLane
final_40_features = features_40_tensor.squeeze(0).tolist()

print("=== FINAL RE-MAPPING COMPLETE ===")
print(f"Original Vector Size : {features_64_tensor.shape[1]} features")
print(f"Mapped Vector Size   : {len(final_40_features)} features")
print("\nYour 40 features are ready for the 10-layer re-uploading loop:")
print(final_40_features)

=== FINAL RE-MAPPING COMPLETE ===
Original Vector Size : 64 features
Mapped Vector Size   : 40 features

Your 40 features are ready for the 10-layer re-uploading loop:
[-0.005518436431884766, 0.045831888914108276, 0.03205948323011398, 0.009530458599328995, -0.0117436982691288, -0.11789628118276596, -0.040757130831480026, 0.1323535293340683, -0.00261569581925869, -0.08158674836158752, -0.02353031374514103, -0.07802815735340118, 0.08688566088676453, 0.0680357813835144, 0.060457609593868256, 0.0691850334405899, -0.09039144217967987, -0.1250300258398056, 0.05793387442827225, -0.006546876858919859, -0.01913169026374817, -0.09480035305023193, 0.05746350809931755, 0.027182545512914658, -0.02398405410349369, 0.1461668163537979, 0.08437196165323257, 0.0073684826493263245, 0.09387184679508209, -0.0786009132862091, 0.008045490831136703, -0.11801117658615112, -0.04813970625400543, 0.07011036574840546, 0.07587683200836182, -0.12751784920692444, 0.1720648854970932, -0.10785749554634094, -0.035471227

In [18]:
import pennylane as qml
import torch

# Define constants
num_qubits = 4
num_layers = 10

# Create the offline qubit simulator device
dev = qml.device("default.qubit", wires=num_qubits)

# Convert your 40 features list back into a PyTorch tensor
quantum_input_features = torch.tensor(final_40_features, device=device)

# Initialize random trainable weights for the 10 layers (3 rotation angles per qubit per layer)
quantum_weights = torch.randn(num_layers, num_qubits, 3, requires_grad=True, device=device)

In [22]:
@qml.qnode(dev, interface="torch")
def data_reuploading_circuit(features, weights):
    # Reshape our flat 40 features into a grid of [10 layers, 4 qubits]
    feature_grid = features.reshape(num_layers, num_qubits)
    
    for layer in range(num_layers):
        # 1. Data Re-uploading Step (CHANGED TO RY: rotates along the real axis)
        for qubit in range(num_qubits):
            qml.RY(feature_grid[layer, qubit], wires=qubit)
            
        # 2. Trainable Variational Step (Applies 3 adjustable angles per qubit)
        for qubit in range(num_qubits):
            qml.Rot(weights[layer, qubit, 0], 
                    weights[layer, qubit, 1], 
                    weights[layer, qubit, 2], 
                    wires=qubit)
            
        # 3. Entanglement Step (Connects the qubits using CNOT gates)
        for qubit in range(num_qubits - 1):
            qml.CNOT(wires=[qubit, qubit + 1])
        if num_qubits > 2:
            qml.CNOT(wires=[num_qubits - 1, 0]) # Wrap around CNOT to close the loop
            
    # Measure the spin output along the Z-axis for all 4 qubits
    return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]

In [23]:
# Execute the quantum forward pass
quantum_output = data_reuploading_circuit(quantum_input_features, quantum_weights)

print("=== RAW QUANTUM CIRCUIT OUTPUTS ===")
print("Your 4-qubit circuit has generated 4 continuous features bounded between -1.0 and 1.0:\n")
for i, val in enumerate(quantum_output):
    print(f"Qubit {i} Matrix Output <Z>: {val.item():.6f}")

=== RAW QUANTUM CIRCUIT OUTPUTS ===
Your 4-qubit circuit has generated 4 continuous features bounded between -1.0 and 1.0:

Qubit 0 Matrix Output <Z>: -0.139657
Qubit 1 Matrix Output <Z>: 0.173354
Qubit 2 Matrix Output <Z>: 0.156531
Qubit 3 Matrix Output <Z>: 0.159048


In [24]:
print("=== VISUAL QUANTUM CIRCUIT DIAGRAM ===")

# qml.draw prints out the schematic of gates step-by-step
circuit_diagram = qml.draw(data_reuploading_circuit)(quantum_input_features, quantum_weights)
print(circuit_diagram)

=== VISUAL QUANTUM CIRCUIT DIAGRAM ===
0: ──RY(-0.01)──Rot(-0.32,1.75,0.52)──╭●───────╭X──RY(-0.01)──Rot(1.22,1.35,-0.40)──╭●───────╭X ···
1: ──RY(0.05)───Rot(0.28,-1.45,-0.01)─╰X─╭●────│───RY(-0.12)──Rot(-1.28,0.32,0.47)──╰X─╭●────│─ ···
2: ──RY(0.03)───Rot(-0.00,0.72,-1.42)────╰X─╭●─│───RY(-0.04)──Rot(-0.95,-0.65,0.81)────╰X─╭●─│─ ···
3: ──RY(0.01)───Rot(0.09,-0.91,-0.86)───────╰X─╰●──RY(0.13)───Rot(-0.16,0.26,-0.11)───────╰X─╰● ···

0: ··· ──RY(-0.00)──Rot(1.05,2.04,1.61)────╭●───────╭X──RY(0.09)──Rot(-0.59,0.07,0.29)─╭●────── ···
1: ··· ──RY(-0.08)──Rot(-0.27,-0.08,-2.34)─╰X─╭●────│───RY(0.07)──Rot(-0.07,0.55,1.78)─╰X─╭●─── ···
2: ··· ──RY(-0.02)──Rot(-1.48,0.02,-0.49)─────╰X─╭●─│───RY(0.06)──Rot(0.70,-0.07,0.30)────╰X─╭● ···
3: ··· ──RY(-0.08)──Rot(0.87,0.86,-1.42)─────────╰X─╰●──RY(0.07)──Rot(0.03,-0.17,1.92)───────╰X ···

0: ··· ─╭X──RY(-0.09)──Rot(-0.26,0.03,-1.08)─╭●───────╭X──RY(-0.02)──Rot(1.08,-0.44,-0.37)─╭●─── ···
1: ··· ─│───RY(-0.13)──Rot(-0.83,0.81,-0.15)─╰X─╭●────│───

In [26]:
import torch
import torch.nn as nn
import pennylane as qml

# =====================================================================
# STEP 1: DEFINE THE COMPLETE CLASSICAL COMPRESSION PIPELINE (1000 -> 40)
# =====================================================================

# 1.1 The Main 4-Layer MLP (1000 -> 64)
class FeatureReducerMLP(nn.Module):
    def __init__(self):
        super(FeatureReducerMLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1000, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
    def forward(self, x):
        return self.mlp(x)

# 1.2 The Bridge Mapper (64 -> 40)
class QuantumBridge(nn.Module):
    def __init__(self):
        super(QuantumBridge, self).__init__()
        self.bridge = nn.Linear(64, 40)
    def forward(self, x):
        return self.bridge(x)

# Instantiate modules completely offline
mlp_reducer = FeatureReducerMLP().to(device)
quantum_mapper = QuantumBridge().to(device)

mlp_reducer.eval()
quantum_mapper.eval()


# =====================================================================
# STEP 2: DEFINE THE SIMPLIFIED PAPER-STYLE QUANTUM CIRCUIT (40 Features)
# =====================================================================
num_qubits = 4
num_layers = 10
dev = qml.device("default.qubit", wires=num_qubits)

@qml.qnode(dev, interface="torch")
def paper_style_circuit(features, weights):
    # Reshape our flat 40 features and 40 weights into [10, 4] grids
    feature_grid = features.reshape(num_layers, num_qubits)
    weight_grid = weights.reshape(num_layers, num_qubits)
    
    for layer in range(num_layers):
        # 1. Combined RY Data Mapping: Scale features directly by variational weights
        for qubit in range(num_qubits):
            combined_angle = feature_grid[layer, qubit] * weight_grid[layer, qubit]
            qml.RY(combined_angle, wires=qubit)
            
        # 2. Pure Entanglement Layer (CNOTs)
        for qubit in range(num_qubits - 1):
            qml.CNOT(wires=[qubit, qubit + 1])
        if num_qubits > 2:
            qml.CNOT(wires=[num_qubits - 1, 0]) # Close the loop
            
    # Measure the PauliZ expectations for our final 4 outputs
    return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]


# =====================================================================
# STEP 3: RUN THE FORWARD PASS FROM IMAGE TO QUANTUM EMBEDDING
# =====================================================================

# Match your raw input tensor to the expected batch dimension format [1, 1000]
mlp_input = probabilities.unsqueeze(0).to(device)

with torch.no_grad():
    # Pass 1: 1,000 features down to 64
    features_64 = mlp_reducer(mlp_input)
    
    # Pass 2: 64 features down to 40
    features_40 = quantum_mapper(features_64)
    
    # Flatten out the batch dimension to feed straight into PennyLane
    quantum_input_features = features_40.squeeze(0)

# Initialize 40 trainable quantum weights (1 unique weight per feature injection)
paper_weights = torch.randn(40, requires_grad=True, device=device)

# Pass 3: Execute your paper-style quantum architecture forward pass
quantum_output = paper_style_circuit(quantum_input_features, paper_weights)


# =====================================================================
# STEP 4: VERIFY AND DISPLAY RESULTS
# =====================================================================
print("=== PIPELINE PIPING COMPLETE ===")
print(f"1. CaiT Vector Ingested       : {mlp_input.shape}")
print(f"2. Classical MLP Reduction     : {features_64.shape}")
print(f"3. Quantum Bridge Mapping      : {features_40.shape}")
print(f"4. Fed into Quantum Circuit   : {quantum_input_features.shape[0]} processed features\n")

print("=== CHIP OUTPUT MATRICES ===")
for i, val in enumerate(quantum_output):
    print(f"Qubit {i} Final Expectation Value <Z>: {val.item():.6f}")

=== PIPELINE PIPING COMPLETE ===
1. CaiT Vector Ingested       : torch.Size([1, 1000])
2. Classical MLP Reduction     : torch.Size([1, 64])
3. Quantum Bridge Mapping      : torch.Size([1, 40])
4. Fed into Quantum Circuit   : 40 processed features

=== CHIP OUTPUT MATRICES ===
Qubit 0 Final Expectation Value <Z>: 0.983021
Qubit 1 Final Expectation Value <Z>: 0.983836
Qubit 2 Final Expectation Value <Z>: 0.982237
Qubit 3 Final Expectation Value <Z>: 0.987107
